# Compare Spyndex Bands

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import spyndex


In [ ]:
for key, band in spyndex.bands.items():
    if not hasattr(band, "sentinel2a"):
        continue
    print(
        f"{band.sentinel2a.band}: {band.common_name} ({band.sentinel2a.wavelength} nm {band.sentinel2a.bandwidth} nm)"
    )

In [ ]:
for key, band in spyndex.bands.items():
    if not hasattr(band, "landsat8"):
        continue
    print(f"{band.landsat8.band}: {band.common_name} ({band.landsat8.wavelength} nm {band.landsat8.bandwidth} nm)")

In [ ]:
for key, band in spyndex.bands.items():
    if not hasattr(band, "landsat9"):
        continue
    print(f"{band.landsat9.band}: {band.common_name} ({band.landsat9.wavelength} nm {band.landsat9.bandwidth} nm)")

In [ ]:
for key, band in spyndex.bands.items():
    if not hasattr(band, "landsat7"):
        continue
    print(f"{band.landsat7.band}: {band.common_name} ({band.landsat7.wavelength} nm {band.landsat7.bandwidth} nm)")

In [ ]:
# Collect band data for all satellites
satellites = ["sentinel2a", "landsat5", "landsat7", "landsat8", "landsat9"]
band_data = []

for key, band in spyndex.bands.items():
    for satellite in satellites:
        if hasattr(band, satellite):
            sat_band = getattr(band, satellite)
            band_data.append(
                {
                    "satellite": satellite,
                    "band_id": sat_band.band,
                    "common_name": band.common_name,
                    "wavelength": sat_band.wavelength,
                    "bandwidth": sat_band.bandwidth,
                }
            )

df = pd.DataFrame(band_data)
df = df.sort_values(["satellite", "wavelength"])
print(f"Total bands: {len(df)}")
df

In [ ]:
# Create a plot showing bands for all satellites with broken axis
fig, (ax1, ax2) = plt.subplots(
    1, 2, figsize=(24, 10), sharey=True, gridspec_kw={"width_ratios": [3, 1], "wspace": 0.05}
)

# Color palette by band type (common name)
band_colors = {
    "blue": "#0000FF",
    "green": "#00CC00",
    "red": "#FF0000",
    "rededge": "#FF6600",
    "nir": "#8B0000",
    "nir08": "#8B0000",
    "nir09": "#660000",
    "swir16": "#8B4513",
    "swir22": "#654321",
    "coastal": "#00BFFF",
    "pan": "#808080",
    "cirrus": "#E0FFFF",
    "lwir": "#4B0082",
    "lwir11": "#4B0082",
    "lwir12": "#3A006F",
    "vre": "#FF8C00",
    "nnir": "#A52A2A",
    "yellowred": "#FF4500",
}

# Plot each band as a horizontal bar showing the wavelength range
y_pos = 0
y_positions = {}
y_labels = []

# Determine the wavelength ranges
lower_max = 2500  # Visible to SWIR
upper_min = 10000  # Thermal bands

for satellite in satellites:
    sat_df = df[df["satellite"] == satellite]
    y_labels.append(satellite.upper())
    y_positions[satellite] = y_pos

    for idx, row in sat_df.iterrows():
        # Calculate the wavelength range (center ± half bandwidth)
        left = row["wavelength"] - row["bandwidth"] / 2
        right = row["wavelength"] + row["bandwidth"] / 2
        width = row["bandwidth"]

        # Get color based on common name
        color = band_colors.get(row["common_name"], "#999999")

        # Decide which axis to plot on
        if row["wavelength"] < lower_max:
            ax = ax1
        else:
            ax = ax2

        # Plot the band as a horizontal bar
        ax.barh(y_pos, width, left=left, height=0.6, color=color, alpha=0.7, edgecolor="black", linewidth=0.5)

        # Add band label with both band ID and common name
        label_text = f"{row['band_id']}\n{row['common_name']}"
        ax.text(
            row["wavelength"],
            y_pos,
            label_text,
            ha="center",
            va="center",
            fontsize=6,
            fontweight="bold",
            clip_on=True,
        )

    y_pos += 1

# Configure left axis (visible to SWIR)
ax1.set_xlim(300, lower_max)
ax1.set_xlabel("Wavelength (nm)", fontsize=12, fontweight="bold")
ax1.set_ylabel("Satellite", fontsize=12, fontweight="bold")
ax1.grid(True, axis="x", alpha=0.3, linestyle="--")
ax1.set_yticks(list(y_positions.values()))
ax1.set_yticklabels(y_labels, fontsize=11)

# Configure right axis (thermal)
ax2.set_xlim(upper_min, 13000)
ax2.set_xlabel("Wavelength (nm)", fontsize=12, fontweight="bold")
ax2.grid(True, axis="x", alpha=0.3, linestyle="--")
ax2.tick_params(labelleft=False)

# Add break marks
d = 0.015  # size of diagonal lines
kwargs = dict(transform=ax1.transAxes, color="k", clip_on=False, linewidth=1)
ax1.plot((1 - d, 1 + d), (-d, +d), **kwargs)
ax1.plot((1 - d, 1 + d), (1 - d, 1 + d), **kwargs)

kwargs = dict(transform=ax2.transAxes, color="k", clip_on=False, linewidth=1)
ax2.plot((-d, +d), (-d, +d), **kwargs)
ax2.plot((-d, +d), (1 - d, 1 + d), **kwargs)

# Add title to the figure
fig.suptitle("Spectral Bands Comparison: Sentinel-2A and Landsat Satellites", fontsize=14, fontweight="bold", y=0.98)

# Add legend based on unique band types present in the data
from matplotlib.patches import Patch

unique_bands = df["common_name"].unique()
legend_elements = [
    Patch(facecolor=band_colors.get(band, "#999999"), alpha=0.7, edgecolor="black", label=band.upper())
    for band in sorted(unique_bands)
]
ax2.legend(handles=legend_elements, loc="upper right", fontsize=9, ncol=1)

plt.tight_layout()
plt.show()